# 4K → 1080p, Drive to Drive

Runs in Google's cloud. Nothing downloads to your computer.

1. Runtime → Change runtime type → **T4 GPU** (optional, roughly 5x faster).
2. Run cell 1, approve the Drive access prompt.
3. Paste the folder path in cell 2 and run it.

What it does: for every video whose long edge is above 1920, writes a 1080p H.264 copy next to it and moves the original into a `4K originals` subfolder. Files already at 1080p or below are left alone. Vertical clips stay vertical.

If a file was already renamed to the convention with a `_3840` or `_2160` suffix, the copy gets `_1080p` in its place. Otherwise the copy keeps the original name plus `_1080p`, ready for the renamer.

In [ ]:
# Cell 1: mount Drive and check ffmpeg
from google.colab import drive
drive.mount('/content/drive')
import subprocess, shutil
print('ffmpeg:', shutil.which('ffmpeg'))
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none (CPU encode, slower but fine)')

In [ ]:
# Cell 2: convert
# Paste the folder as it appears under /content/drive/MyDrive/ (or /content/drive/Shareddrives/<drive>/ for shared drives).
# Tip: run  !ls "/content/drive/MyDrive"  in a new cell to find it.
FOLDER = '/content/drive/MyDrive/tails.com/30.06.25_PeeInThePark/Raw Clips/Card 1'

RECURSIVE = False          # True to also process subfolders
CRF = 18                   # quality, 18 is visually lossless-ish, 23 is smaller
MOVE_ORIGINALS = True      # move 4K sources into '4K originals' inside each folder

import os, re, json, subprocess, shutil, time
from pathlib import Path

VIDEO_EXT = {'.mp4', '.mov', '.m4v', '.mkv', '.avi', '.mts'}
use_nvenc = bool(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip())

def probe(p):
    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                        '-show_entries', 'stream=width,height:stream_tags=rotate:side_data=rotation',
                        '-of', 'json', str(p)], capture_output=True, text=True)
    try:
        s = json.loads(r.stdout)['streams'][0]
    except Exception:
        return None
    w, h = int(s.get('width', 0)), int(s.get('height', 0))
    rot = int(s.get('tags', {}).get('rotate', 0) or 0)
    for sd in s.get('side_data_list', []) or []:
        if 'rotation' in sd: rot = int(sd['rotation'])
    if abs(rot) % 180 == 90: w, h = h, w   # display orientation
    return w, h

def target_name(p):
    stem, ext = p.stem, '.mp4'
    m = re.match(r'^(\d{4}_.+?_.+?_.+?_\d{2,3})_(3840|2160|4K|4k|3840px|2160px)$', stem)
    if m: return m.group(1) + '_1080p' + ext
    return stem + '_1080p' + ext

def convert(src, dst):
    vf = "scale='if(gt(iw,ih),1920,-2)':'if(gt(iw,ih),-2,1920)'"
    if use_nvenc:
        venc = ['-c:v', 'h264_nvenc', '-preset', 'p5', '-rc', 'vbr', '-cq', str(CRF + 1), '-b:v', '0']
    else:
        venc = ['-c:v', 'libx264', '-preset', 'medium', '-crf', str(CRF)]
    cmd = ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', str(src), '-vf', vf,
           *venc, '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart', str(dst)]
    return subprocess.run(cmd, capture_output=True, text=True)

root = Path(FOLDER)
assert root.is_dir(), f'Folder not found: {root}'
files = [p for p in (root.rglob('*') if RECURSIVE else root.iterdir())
         if p.is_file() and p.suffix.lower() in VIDEO_EXT and '4K originals' not in p.parts]
print(f'{len(files)} video(s) found. Encoder: {"NVENC (GPU)" if use_nvenc else "libx264 (CPU)"}')

done = skipped = failed = 0
for p in sorted(files):
    dims = probe(p)
    if not dims:
        print(f'  ? could not read {p.name}'); failed += 1; continue
    w, h = dims
    if max(w, h) <= 1920:
        print(f'  = {p.name} already {w}x{h}, skipped'); skipped += 1; continue
    dst = p.with_name(target_name(p))
    if dst.exists():
        print(f'  = {dst.name} exists, skipped'); skipped += 1; continue
    t = time.time()
    print(f'  > {p.name} {w}x{h} → {dst.name} ...', end=' ', flush=True)
    r = convert(p, dst)
    if r.returncode != 0 or not dst.exists():
        print('FAILED'); print(r.stderr[-400:]); failed += 1
        if dst.exists(): dst.unlink()
        continue
    if MOVE_ORIGINALS:
        keep = p.parent / '4K originals'; keep.mkdir(exist_ok=True)
        shutil.move(str(p), str(keep / p.name))
    print(f'ok ({time.time() - t:.0f}s)'); done += 1

print(f'\nconverted {done}, skipped {skipped}, failed {failed}')
drive.flush_and_unmount() if False else None  # Drive syncs automatically; leave mounted